# Chien luoc Close > MA10 thi Buy va MA10 > MA20; 
### Close < MA10 va MA10 < 20 thi Sell
=> Làm sao để code được

### Buoc 1: Lay data
### Buoc 2: Xu ly data
### Buoc 2.1, 2.2: Buy_Signal, Sell_Signal
### Buoc 3: Show len chart
### Buoc 4: Day len Redis

# Buoc 1: Load data

In [1]:
import pandas as pd
from datetime import datetime, timedelta
import ccxt # 100 san

# Khởi tạo kết nối đến sàn Binance
exchange = ccxt.binance() # Ko can API, Secret Key

# Định dạng ngày tháng
symbol = 'NEARUSDT'
from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hôm nay
print(f'from_date: {from_date}')
to_date = (datetime.now() + timedelta(days=0)).strftime('%Y-%m-%d')
timeframe = '1m' # 1m

since = exchange.parse8601(from_date + 'T00:00:00Z') # Chuyển đổi từ ngày sang timestamp
print(f'since: {since}')
end = exchange.parse8601(to_date + 'T00:00:00Z')

# Lấy dữ liệu thị trường từ sàn Binance
# 1m docs.ccxt.com/#/?id=fetch-ohlcv
ohlcv = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=1000)

# Chuyển dữ liệu thành DataFrame
data = pd.DataFrame(ohlcv, columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
data['Timestamp'] = pd.to_datetime(data['Timestamp'], unit='ms')  # Chuyển đổi timestamp từ milliseconds sang datetime       
data = data.rename(columns={'Timestamp': 'Datetime'}) # Đổi tên cột Timestamp thành Datetime
data

from_date: 2025-07-24
since: 1753315200000


,Datetime,Open,High,Low,Close,Volume
0,2025-07-24 00:00:00,2.755,2.755,2.746,2.747,15351.8
1,2025-07-24 00:01:00,2.747,2.749,2.746,2.747,4646.2
2,2025-07-24 00:02:00,2.747,2.748,2.745,2.745,6270.5
3,2025-07-24 00:03:00,2.745,2.745,2.742,2.743,8516.5
4,2025-07-24 00:04:00,2.743,2.744,2.742,2.742,5047.4
...,...,...,...,...,...,...
911,2025-07-24 15:11:00,2.816,2.816,2.804,2.811,27253.7
912,2025-07-24 15:12:00,2.811,2.819,2.811,2.817,13549.8
913,2025-07-24 15:13:00,2.817,2.824,2.817,2.823,15795.8
914,2025-07-24 15:14:00,2.822,2.825,2.820,2.824,21257.4


# Buoc 2: Xu ly
### Tinh toan them MA10

## Xu ly: Tinh toan ra chi bao

In [2]:
import talib as ta

In [3]:
data['MA10'] = ta.SMA(data['Close'], timeperiod=10)

In [6]:
data.head(10)

,Datetime,Open,High,Low,Close,Volume,MA10
0,2025-07-24 00:00:00,2.755,2.755,2.746,2.747,15351.8,NaN
1,2025-07-24 00:01:00,2.747,2.749,2.746,2.747,4646.2,NaN
2,2025-07-24 00:02:00,2.747,2.748,2.745,2.745,6270.5,NaN
3,2025-07-24 00:03:00,2.745,2.745,2.742,2.743,8516.5,NaN
4,2025-07-24 00:04:00,2.743,2.744,2.742,2.742,5047.4,NaN
5,2025-07-24 00:05:00,2.742,2.742,2.735,2.736,8531.2,NaN
6,2025-07-24 00:06:00,2.736,2.738,2.733,2.737,8281.7,NaN
7,2025-07-24 00:07:00,2.737,2.745,2.736,2.745,17095.4,NaN
8,2025-07-24 00:08:00,2.745,2.745,2.741,2.742,3010.5,NaN
9,2025-07-24 00:09:00,2.742,2.743,2.739,2.740,7131.3,2.7424


### Xu ly de tinh Buy_Signal va Sell_Signal

In [7]:
# Tin hieu Buy xay ra khi gia dong cua Close tren MA10
data['Buy_Signal'] = (data['Close'] > data['MA10'])

# Tin hieu Sell xay ra khi gia dong cua Close duoi MA10
data['Sell_Signal'] = (data['Close'] < data['MA10'])

In [8]:
data.head(10)

,Datetime,Open,High,Low,Close,Volume,MA10,Buy_Signal,Sell_Signal
0,2025-07-24 00:00:00,2.755,2.755,2.746,2.747,15351.8,NaN,False,False
1,2025-07-24 00:01:00,2.747,2.749,2.746,2.747,4646.2,NaN,False,False
2,2025-07-24 00:02:00,2.747,2.748,2.745,2.745,6270.5,NaN,False,False
3,2025-07-24 00:03:00,2.745,2.745,2.742,2.743,8516.5,NaN,False,False
4,2025-07-24 00:04:00,2.743,2.744,2.742,2.742,5047.4,NaN,False,False
5,2025-07-24 00:05:00,2.742,2.742,2.735,2.736,8531.2,NaN,False,False
6,2025-07-24 00:06:00,2.736,2.738,2.733,2.737,8281.7,NaN,False,False
7,2025-07-24 00:07:00,2.737,2.745,2.736,2.745,17095.4,NaN,False,False
8,2025-07-24 00:08:00,2.745,2.745,2.741,2.742,3010.5,NaN,False,False
9,2025-07-24 00:09:00,2.742,2.743,2.739,2.740,7131.3,2.7424,False,True


# Day sang Redis

In [9]:
# Them symbol vao DataFrame
data['Symbol'] = symbol

In [10]:
data

,Datetime,Open,High,Low,Close,Volume,MA10,Buy_Signal,Sell_Signal,Symbol
0,2025-07-24 00:00:00,2.755,2.755,2.746,2.747,15351.8,NaN,False,False,NEARUSDT
1,2025-07-24 00:01:00,2.747,2.749,2.746,2.747,4646.2,NaN,False,False,NEARUSDT
2,2025-07-24 00:02:00,2.747,2.748,2.745,2.745,6270.5,NaN,False,False,NEARUSDT
3,2025-07-24 00:03:00,2.745,2.745,2.742,2.743,8516.5,NaN,False,False,NEARUSDT
4,2025-07-24 00:04:00,2.743,2.744,2.742,2.742,5047.4,NaN,False,False,NEARUSDT
...,...,...,...,...,...,...,...,...,...,...
911,2025-07-24 15:11:00,2.816,2.816,2.804,2.811,27253.7,2.8090,True,False,NEARUSDT
912,2025-07-24 15:12:00,2.811,2.819,2.811,2.817,13549.8,2.8110,True,False,NEARUSDT
913,2025-07-24 15:13:00,2.817,2.824,2.817,2.823,15795.8,2.8129,True,False,NEARUSDT
914,2025-07-24 15:14:00,2.822,2.825,2.820,2.824,21257.4,2.8152,True,False,NEARUSDT


# Khai bao de day qua Redis

In [11]:
import redis
redis = redis.Redis(host='localhost', port=6379, db=15) # 0-5 ; 6-10 ; 11-15
hash_key = 'CRT_Binance MA10_K12' + symbol

last_record = data.iloc[-1] # Cay nen cuoi cung

if (last_record['Buy_Signal'] == True) | (last_record['Sell_Signal'] == True):
    print(last_record)
    print('Có tín hiệu giao dịch')
    
    # Day nguyen last_record vao redis
    redis.set(hash_key, last_record.to_json())
else:
    print('Không có tín hiệu giao dịch')

Datetime       2025-07-24 15:15:00
Open                         2.823
High                         2.824
Low                          2.823
Close                        2.824
Volume                       652.2
MA10                        2.8175
Buy_Signal                    True
Sell_Signal                  False
Symbol                    NEARUSDT
Name: 915, dtype: object
Có tín hiệu giao dịch
